# 08b — Day 8b Evaluation: Embedding-Family Fusion (HuBERT + WavLM + TRILLsson)

**Standalone**, same pattern as every other evaluation notebook — duplicated helper functions.
**Independent of Day 8a**: this tests a different variable (fusing embedding *families*) using each
embedding's plain, original pipeline (no augmentation, no session-fusion) — deliberately isolated
from Day 8a's augmentation question, same one-variable-at-a-time discipline used since Day 1. If
Day 8a's augmented WavLM result is strong, fusing the *augmented* pipelines is Day 10's job, not
this notebook's.

**What this tests:** does combining information across embedding families beat any single one
alone? Three confirmed/logged single-embedding results going in:
- HuBERT L12 (Day 3): AUC 0.611, CI [0.551, 0.683] — confirmed, permutation p=0.01
- TRILLsson (Day 4): AUC 0.590, CI [0.535, 0.651]
- WavLM L6 (Day 7): AUC 0.663, CI [0.600, 0.710] — current single-embedding leader

**Two fusion strategies, isolated from each other:**
1. **Feature-level concatenation** — stack the embeddings into one wide vector per recording, feed
   through the same PCA-capable pipeline machinery Day 1/6 already built. Tested at 2-way
   (HuBERT+WavLM, the two strongest) and 3-way (+TRILLsson), with and without PCA (dimensionality
   gets large: 3072 dims at 2-way, 4096 at 3-way, against ~260 training rows/fold).
2. **OOF-probability averaging** — average each embedding's own already-computed, leak-free
   out-of-fold prediction scores, no new classifier trained on top (avoids a second overfitting
   surface on 14 speakers). Also tested at 2-way and 3-way.

**No permutation test here** — reserved for Day 10's single finalized candidate, same discipline as
Day 8a.

**Inputs needed (Kaggle "Add Input") — 4 total:**
1. The **`Fatigue_data`** dataset — for `labels.csv`.
2. **Day 3's `01_features.ipynb`** output — for `hubert_base_all_layers.parquet`.
3. **Day 4's `04_features_trillsson.ipynb`** output — for `trillsson.parquet`.
4. **Day 7's `07_features_wavlm.ipynb`** output — for `wavlm_base_plus_all_layers.parquet`.

## 0. Setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)

RANDOM_SEED = 42
HUBERT_LAYER = 12
WAVLM_LAYER = 6

LABELS_CSV = next(Path("/kaggle/input").rglob("labels.csv"))
HUBERT_PARQUET = next(Path("/kaggle/input").rglob("hubert_base_all_layers.parquet"))
TRILLSSON_PARQUET = next(Path("/kaggle/input").rglob("trillsson.parquet"))
WAVLM_PARQUET = next(Path("/kaggle/input").rglob("wavlm_base_plus_all_layers.parquet"))

print(f"LABELS_CSV        = {LABELS_CSV}")
print(f"HUBERT_PARQUET    = {HUBERT_PARQUET}")
print(f"TRILLSSON_PARQUET = {TRILLSSON_PARQUET}")
print(f"WAVLM_PARQUET     = {WAVLM_PARQUET}")

## 1. Load & Merge

Builds one unified dataframe with all three embedding families aligned to the same row order, so
`X_hubert[i]`, `X_wavlm[i]`, and `X_trillsson[i]` always refer to the same recording.

In [ ]:
labels = pd.read_csv(LABELS_CSV)
hubert_all = pd.read_parquet(HUBERT_PARQUET)
trillsson = pd.read_parquet(TRILLSSON_PARQUET)
wavlm_all = pd.read_parquet(WAVLM_PARQUET)

hubert_cols = [c for c in hubert_all.columns if c.startswith(f"layer{HUBERT_LAYER}_emb_")]
assert len(hubert_cols) == 1536, f"expected 1536 HuBERT layer-{HUBERT_LAYER} columns, got {len(hubert_cols)}"
hubert_df = hubert_all[["filename"] + hubert_cols].copy()
hubert_df.columns = ["filename"] + [f"hubert_emb_{i}" for i in range(1536)]

wavlm_cols = [c for c in wavlm_all.columns if c.startswith(f"layer{WAVLM_LAYER}_emb_")]
assert len(wavlm_cols) == 1536, f"expected 1536 WavLM layer-{WAVLM_LAYER} columns, got {len(wavlm_cols)}"
wavlm_df = wavlm_all[["filename"] + wavlm_cols].copy()
wavlm_df.columns = ["filename"] + [f"wavlm_emb_{i}" for i in range(1536)]

trillsson_cols = [c for c in trillsson.columns if c.startswith("emb_")]
assert len(trillsson_cols) == 1024, f"expected 1024 TRILLsson columns, got {len(trillsson_cols)}"
trillsson_df = trillsson[["filename"] + trillsson_cols].copy()
trillsson_df.columns = ["filename"] + [f"trillsson_emb_{i}" for i in range(1024)]

df = labels.merge(hubert_df, on="filename", how="inner") \
           .merge(wavlm_df, on="filename", how="inner") \
           .merge(trillsson_df, on="filename", how="inner")
assert len(df) == len(labels) == 280, f"expected 280 rows after merge, got {len(df)}"

LABEL_MAP = {"non_fatigued": 0, "fatigued": 1}
df["y"] = df["label"].map(LABEL_MAP)
assert df["y"].isna().sum() == 0

y = df["y"].to_numpy()
groups = df["participant_id"].to_numpy()

hubert_emb_cols = [c for c in df.columns if c.startswith("hubert_emb_")]
wavlm_emb_cols = [c for c in df.columns if c.startswith("wavlm_emb_")]
trillsson_emb_cols = [c for c in df.columns if c.startswith("trillsson_emb_")]

X_hubert = df[hubert_emb_cols].to_numpy()
X_wavlm = df[wavlm_emb_cols].to_numpy()
X_trillsson = df[trillsson_emb_cols].to_numpy()

print(f"X_hubert:    {X_hubert.shape}")
print(f"X_wavlm:     {X_wavlm.shape}")
print(f"X_trillsson: {X_trillsson.shape}")
print(f"{len(np.unique(groups))} participants, all row-aligned across the three embedding families.")

## 2. Candidates & Nested CV (duplicated from `02_train_evaluate.ipynb` / `06_evaluate_combined.ipynb`)

Same `nested_loso_cv_variant` mechanism as Day 1/6 — candidates as `(label, pipeline)` pairs from an
arbitrary generator, so PCA slots in as just another pipeline step. No augmentation-awareness needed
here (no `is_original` masking) since every embedding used in this notebook is the plain, original,
unperturbed version.

In [ ]:
CANDIDATE_CS = [0.001, 0.01, 0.1, 1.0, 10.0]


def build_pipeline_pca(model_name: str, c: float, pca_components: int | None = None) -> Pipeline:
    steps = [("scaler", StandardScaler())]
    if pca_components is not None:
        steps.append(("pca", PCA(n_components=pca_components, random_state=RANDOM_SEED)))
    if model_name == "logreg":
        clf = LogisticRegression(C=c, max_iter=5000, class_weight="balanced")
    elif model_name == "linsvc":
        clf = LinearSVC(C=c, max_iter=10000, class_weight="balanced")
    else:
        raise ValueError(model_name)
    steps.append(("clf", clf))
    return Pipeline(steps)


def make_candidates_plain():
    return [
        (f"{name}_C={c}", build_pipeline_pca(name, c, pca_components=None))
        for name in ["logreg", "linsvc"]
        for c in CANDIDATE_CS
    ]


def make_candidates_pca(pca_options=(10, 30, 50)):
    return [
        (f"{name}_C={c}_pca={k}", build_pipeline_pca(name, c, pca_components=k))
        for name in ["logreg", "linsvc"]
        for c in CANDIDATE_CS
        for k in pca_options
    ]


def score_fn(fitted_pipeline: Pipeline, X_in: np.ndarray) -> np.ndarray:
    clf = fitted_pipeline.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        return fitted_pipeline.predict_proba(X_in)[:, 1]
    return fitted_pipeline.decision_function(X_in)


def best_threshold(scores: np.ndarray, y_true: np.ndarray) -> float:
    best_t, best_ba = 0.5, -1.0
    for t in np.unique(scores):
        ba = balanced_accuracy_score(y_true, (scores >= t).astype(int))
        if ba > best_ba:
            best_ba, best_t = ba, t
    return best_t

In [ ]:
def nested_loso_cv_variant(X, y, groups, make_candidates_fn, n_inner_splits=6):
    participants = np.unique(groups)
    oof_scores = np.zeros(len(y), dtype=float)
    oof_preds = np.zeros(len(y), dtype=int)
    fold_records = []
    n_outer_checked, n_inner_checked = 0, 0

    for test_participant in participants:
        outer_test_mask = groups == test_participant
        outer_train_mask = ~outer_test_mask
        assert not (set(groups[outer_train_mask]) & set(groups[outer_test_mask])), \
            f"outer leakage: {test_participant} present in both outer train and test"
        n_outer_checked += 1

        X_tr, y_tr, g_tr = X[outer_train_mask], y[outer_train_mask], groups[outer_train_mask]
        X_te = X[outer_test_mask]

        candidates = make_candidates_fn()
        cand_aucs = {i: [] for i in range(len(candidates))}
        cand_scores = {i: [] for i in range(len(candidates))}
        cand_labels = {i: [] for i in range(len(candidates))}

        gkf = GroupKFold(n_splits=n_inner_splits)
        for inner_tr_idx, inner_val_idx in gkf.split(X_tr, y_tr, groups=g_tr):
            assert not (set(g_tr[inner_tr_idx]) & set(g_tr[inner_val_idx])), \
                f"inner leakage within outer fold {test_participant}"
            n_inner_checked += 1

            X_itr, y_itr = X_tr[inner_tr_idx], y_tr[inner_tr_idx]
            X_ival, y_ival = X_tr[inner_val_idx], y_tr[inner_val_idx]
            if len(np.unique(y_itr)) < 2 or len(np.unique(y_ival)) < 2:
                continue

            for i, (_, pipe) in enumerate(candidates):
                fitted = clone(pipe).fit(X_itr, y_itr)
                s = score_fn(fitted, X_ival)
                cand_aucs[i].append(roc_auc_score(y_ival, s))
                cand_scores[i].append(s)
                cand_labels[i].append(y_ival)

        mean_aucs = {i: np.mean(v) for i, v in cand_aucs.items() if v}
        best_i = max(mean_aucs, key=mean_aucs.get)
        best_label, best_template = candidates[best_i]

        pooled_inner_scores = np.concatenate(cand_scores[best_i])
        pooled_inner_labels = np.concatenate(cand_labels[best_i])
        threshold = best_threshold(pooled_inner_scores, pooled_inner_labels)

        final_pipe = clone(best_template).fit(X_tr, y_tr)
        test_scores = score_fn(final_pipe, X_te)
        oof_scores[outer_test_mask] = test_scores
        oof_preds[outer_test_mask] = (test_scores >= threshold).astype(int)

        fold_records.append({
            "test_participant": test_participant,
            "n_test": int(outer_test_mask.sum()),
            "chosen_label": best_label,
            "inner_cv_auc": mean_aucs[best_i],
            "threshold": threshold,
        })

    print(f"  Verified zero participant overlap: {n_outer_checked} outer folds, "
          f"{n_inner_checked} inner GroupKFold(6) splits.")
    return oof_scores, oof_preds, pd.DataFrame(fold_records)


def bootstrap_auc_ci(participant_ids, y_true, scores, n_boot=1000, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    participants = np.unique(participant_ids)
    idx_by_participant = {p: np.where(participant_ids == p)[0] for p in participants}
    boot_aucs = []
    for _ in range(n_boot):
        sampled = rng.choice(participants, size=len(participants), replace=True)
        idx = np.concatenate([idx_by_participant[p] for p in sampled])
        y_boot, s_boot = y_true[idx], scores[idx]
        if len(np.unique(y_boot)) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_boot, s_boot))
    lo, hi = np.percentile(boot_aucs, [2.5, 97.5])
    return lo, hi, np.array(boot_aucs)


results_log = []
oof_by_embedding = {}


def run_variant(name, X_in, make_candidates_fn):
    print(f"=== {name} ===")
    oof_scores, oof_preds, fold_info = nested_loso_cv_variant(X_in, y, groups, make_candidates_fn)
    auc = roc_auc_score(y, oof_scores)
    ci_lo, ci_hi, _ = bootstrap_auc_ci(groups, y, oof_scores)
    print(f"  Pooled AUC = {auc:.3f}  (95% CI [{ci_lo:.3f}, {ci_hi:.3f}])\n")
    results_log.append({"variant": name, "auc": auc, "ci_lo": ci_lo, "ci_hi": ci_hi})
    return oof_scores

## 3. Single-Embedding Baselines (sanity check + source OOF scores for ensembling)

Reproduces each embedding's already-logged plain result through this notebook's own code path — a
sanity check (should land close to the referenced numbers above) and, at the same time, produces the
leak-free OOF score arrays Section 5's averaging ensemble reuses directly, at no extra cost.

In [ ]:
oof_by_embedding["hubert"] = run_variant("HuBERT L12 alone (sanity check vs. Day 3: 0.611)", X_hubert, make_candidates_plain)
oof_by_embedding["wavlm"] = run_variant("WavLM L6 alone (sanity check vs. Day 7: 0.663)", X_wavlm, make_candidates_plain)
oof_by_embedding["trillsson"] = run_variant("TRILLsson alone (sanity check vs. Day 4: 0.590)", X_trillsson, make_candidates_plain)

## 4. Feature-Level Concatenation

Stack embeddings into one wide vector, same PCA-capable machinery as Day 1/6. Tested with and
without PCA given the dimensionality (3072 dims at 2-way, 4096 at 3-way, against ~260 training
rows/fold — the same `p >> n` concern Day 1 first diagnosed).

In [ ]:
X_concat_2way = np.hstack([X_hubert, X_wavlm])
X_concat_3way = np.hstack([X_hubert, X_wavlm, X_trillsson])
print(f"X_concat_2way (HuBERT+WavLM): {X_concat_2way.shape}")
print(f"X_concat_3way (HuBERT+WavLM+TRILLsson): {X_concat_3way.shape}")

run_variant("Concat: HuBERT + WavLM (no PCA)", X_concat_2way, make_candidates_plain)
run_variant("Concat: HuBERT + WavLM + TRILLsson (no PCA)", X_concat_3way, make_candidates_plain)
run_variant("Concat: HuBERT + WavLM + TRILLsson + PCA (10/30/50)", X_concat_3way, make_candidates_pca)

## 5. OOF-Probability Averaging Ensemble

Averages each embedding's own leak-free OOF scores from Section 3 pointwise -- no new classifier
trained on top, so this introduces no new overfitting surface. Averaging two (or three) already
leak-free per-fold OOF vectors is itself leak-free: each embedding's score for a held-out
participant already came from a model that never saw that participant.

In [ ]:
avg_2way = (oof_by_embedding["hubert"] + oof_by_embedding["wavlm"]) / 2
avg_3way = (oof_by_embedding["hubert"] + oof_by_embedding["wavlm"] + oof_by_embedding["trillsson"]) / 3

for name, scores in [("Average: HuBERT + WavLM", avg_2way),
                     ("Average: HuBERT + WavLM + TRILLsson", avg_3way)]:
    auc = roc_auc_score(y, scores)
    ci_lo, ci_hi, _ = bootstrap_auc_ci(groups, y, scores)
    print(f"{name}: AUC={auc:.3f}  CI=[{ci_lo:.3f}, {ci_hi:.3f}]")
    results_log.append({"variant": name, "auc": auc, "ci_lo": ci_lo, "ci_hi": ci_hi})

## 6. Day 8b Results

In [ ]:
results_df = pd.DataFrame(results_log)
print(results_df.to_string(index=False))

DAY8B_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4",
                "#2a78d6", "#eb6834", "#1baf7a"]
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e1e0d9"

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = np.arange(len(results_df))
ax.barh(y_pos, results_df["auc"], color=DAY8B_COLORS[: len(results_df)], height=0.55, zorder=3)
ax.errorbar(
    results_df["auc"], y_pos,
    xerr=[results_df["auc"] - results_df["ci_lo"], results_df["ci_hi"] - results_df["auc"]],
    fmt="none", ecolor=INK, elinewidth=1, capsize=3, zorder=4,
)
ax.axvline(0.5, color=MUTED, linewidth=1, linestyle="--", zorder=2)
ax.axvline(0.651, color="#e87ba4", linewidth=1, linestyle=":", zorder=2, label="Day 6 final (HuBERT, 0.651)")
ax.axvline(0.663, color="#1baf7a", linewidth=1, linestyle=":", zorder=2, label="Day 7 plain WavLM L6 (0.663)")
ax.set_yticks(y_pos)
ax.set_yticklabels(results_df["variant"], color=INK)
ax.set_xlim(0, 1)
ax.set_xlabel("Pooled ROC-AUC (error bars = 95% bootstrap CI)", color=INK)
ax.tick_params(colors=MUTED)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color=GRID, linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.legend(frameon=False, labelcolor=INK, loc="lower right")
plt.tight_layout()
plt.show()

best_idx = results_df["auc"].idxmax()
print(f"\nBest Day 8b configuration: {results_df.loc[best_idx, 'variant']}  "
      f"AUC={results_df.loc[best_idx, 'auc']:.3f}  "
      f"CI=[{results_df.loc[best_idx, 'ci_lo']:.3f}, {results_df.loc[best_idx, 'ci_hi']:.3f}]")
print("No permutation test run here -- reserved for Day 10's final buffer.")

## How to read Day 8b's results

- **Sanity check first:** the three single-embedding baselines in Section 3 should land close to
  0.611 / 0.663 / 0.590. A large gap on any of them means something changed in this notebook's code
  path -- investigate before trusting the fusion numbers below it.
- **A fusion variant clearly beats WavLM L6 alone (0.663)**, with the CI lower bound moving up too:
  embedding-family fusion is a genuine win, worth carrying into Day 10's shortlist as a candidate to
  combine with Day 8a/9's winning augmentation techniques.
- **Every fusion variant sits at or below the best single embedding (WavLM L6)**: fusion doesn't
  help at this sample size -- plausible if the embeddings are redundant enough (all three are
  ultimately reading the same 280 recordings) that concatenation just adds noise/dimensionality
  without adding real complementary signal, and averaging just pulls a strong predictor (WavLM)
  toward two weaker ones. Report as an honest negative result and drop fusion from Day 10's
  shortlist -- carry Day 7's plain WavLM L6 (and/or Day 8a's result once known) forward instead.
- **Concatenation vs. averaging split** (one helps, the other doesn't): informative on its own --
  averaging already-tuned per-embedding predictions being more robust than concatenation would
  suggest the individual embeddings' *rankings* carry more reliable signal than their raw features
  do once combined into one high-dimensional space at this N.
- **TRILLsson's presence hurting the 3-way variants vs. the 2-way (HuBERT+WavLM) ones**: consistent
  with TRILLsson being the weakest of the three (0.590) -- diluting rather than complementing.

**No permutation test yet.** Update `PROJECT_PLAN.md`'s Day 7+ roadmap section with today's numbers
once Day 8a's real results are also in, so Day 10's shortlist reflects the full Day 7-9 picture.